# Todos os modelos do grupo

As células de cada modelo são as dos notebooks originais, sem alteração. Para trocar de grupo, mude só a célula de **Hiperparâmetros** e rode tudo.

In [1]:
import pandas as pd
import numpy as np
import random
import pyomo.environ as pyo
from pyomo.environ import *
from pyomo.environ import SolverFactory
import yfinance as yf
from pathlib import Path
import os
import matplotlib.pyplot as plt
from pyomo.util.infeasible import log_infeasible_constraints
import logging


In [2]:
anos = ['2016-12-31', '2017-12-31', '2018-12-31', '2019-12-31', '2020-12-31', '2021-12-31', '2022-12-31', '2023-12-31', '2024-12-31', '2025-12-31', '2026-12-31']
print(anos)

['2016-12-31', '2017-12-31', '2018-12-31', '2019-12-31', '2020-12-31', '2021-12-31', '2022-12-31', '2023-12-31', '2024-12-31', '2025-12-31', '2026-12-31']


## LER OS RETORNOS

In [3]:

# Retornos

dfs = {}
for ano in anos:
    ano_usado = ano.split('-')[0]
    ano_usado_mais_um = int(ano_usado)+1
    # print(ano_usado, "-", type(ano_usado))

    nome = f'df_ativos_{ano_usado}.csv'
    # print(nome, "-", type(nome))
    # caminho = Path('retornos_ativos_otimizacao') / nome
    dfs[ano_usado] = pd.read_csv('../../../novos-dados-retornos/otimizacao/'+nome)
    dfs[ano_usado]['date'] = pd.to_datetime(dfs[ano_usado]['date'])
    dfs[ano_usado] = dfs[ano_usado].set_index('date')
    # print(dfs[ano_usado])
    dfs[ano_usado] = dfs[ano_usado].drop(columns=dfs[ano_usado].columns[(dfs[ano_usado] == 0).all()])
    # dfs[ano_usado] = dfs[ano_usado].loc[str(ano_usado_mais_um)]
    ativos_manter = (dfs[ano_usado]==0.0).mean()
    c_manter = ativos_manter[ativos_manter<0.5].index
    dfs[ano_usado] = dfs[ano_usado][c_manter]

    print(ano,'-',len(dfs[ano_usado].columns))
    

2016-12-31 - 68
2017-12-31 - 69
2018-12-31 - 75
2019-12-31 - 77
2020-12-31 - 80
2021-12-31 - 90
2022-12-31 - 98
2023-12-31 - 99
2024-12-31 - 99
2025-12-31 - 99
2026-12-31 - 100


## Hiperparâmetros (vale para todos os modelos)

In [4]:
vb_cardinalidade_max = 15
# vb_cardinalidade_min = 10
vb_peso_maximo = 0.15
# vb_peso_minimo = 0.02
vb_theta = 0.1

# Goal Programming

In [5]:
y = []

carteiras_anuais = {}

melhor_pesos = None
historico = []
carteiras_criadas = []


for an in anos:
    ano = an.split("-")[0]
    print("COMEÇANDO ANO NOVO: ",ano)
    #deletar modelo
    if 'model' in locals():
        del model
        print('Modelo Antigo deletado \n Iniciando Novo')
    else:
        print("Nao consta model")
        pass
    
    # Retornos só para filtrar colunas
    df_usado = dfs[str(ano)]
    retorno_usado = df_usado.copy()    
        #SCORES MF e PIO------------
    score_mf_todos_anos = pd.read_excel(f'../../../score-raspagem/magic_formula/resultado/{int(ano)-1}.xlsx')
    score_mf = score_mf_todos_anos[['Unnamed: 0', 'magic_formula']].set_index('Unnamed: 0').T
    
    score_pio_todos_anos = pd.read_excel(f'../../../score-raspagem/piotroski/resultado/{int(ano)-1}.xlsx')
    score_pio = score_pio_todos_anos[['Unnamed: 0', 'p_norm']].set_index('Unnamed: 0').T
    
    ativos = retorno_usado.columns.intersection(score_pio.columns).intersection(score_mf.columns)
    print("ATIVOS: ",ativos)
    score_usado_mf = score_mf[ativos].copy()
    score_usado_pio = score_pio[ativos].copy()

    print("Score Atualizado: ",int(ano)-1)


    print("-----")
    

    # if df_usado == 'df_ativos_2015':
    #     continue
    # else:
    print(f"{int(ano)-1} TAMANHOS: Retorno {len(ativos)}, Score PIO {len(score_usado_pio.columns)}, Score MF {len(score_usado_mf.columns)}")
    print("# ------ CRIAÇÃO DO MODELO")
    print("## Carteira criada para o ano: ",int(ano))

    # # print("## UTILIZANDO SCORE DO ANO DE: ",ano)
    # print("## UTILIZANDO DADOS DE RETORNO DE: ",str(ano_um))
    # # print("## UTILIZANDO EXCESSO DO ANO DE: ",ano_um)
    # print("## UTILIZANDO SIGMAS DO ANO DE: ",ano_um)

    model = pyo.ConcreteModel()

    #---------VARIÁVEIS-----------
    model.nome_ativos = pyo.Set(initialize = ativos)
    model.ativos = pyo.RangeSet(0, len(ativos)-1)
    # model.dias = pyo.RangeSet(0, len(retorno_usado)-1)
    # model.retornos_ativos = pyo.Param(model.dias, model.ativos, initialize=lambda model,dia, ativo: retorno_usado.iloc[dia, ativo])    
    # model.theta = pyo.Param(initialize=vb_theta)
    model.score_p = pyo.Param(model.ativos, initialize=lambda model,a: score_usado_pio.iloc[0,a])
    model.score_m = pyo.Param(model.ativos, initialize=lambda model,a: score_usado_mf.iloc[0,a])
    model.cardinalidade_valor_max = pyo.Param(initialize=vb_cardinalidade_max)
    # model.cardinalidade_valor_min = pyo.Param(initialize=vb_cardinalidade_min)
    model.peso_maximo = pyo.Param(initialize=vb_peso_maximo)
    # model.peso_minimo = pyo.Param(initialize=vb_peso_minimo)
    model.x = pyo.Var(model.ativos, bounds=(0,1))
    model.y = pyo.Var(model.ativos, within=pyo.Binary)
    # model.excesso = pyo.Param( model.ativos , initialize = lambda model,a: excesso_usado.mean().iloc[a])
    # model.sigma = pyo.Param(model.ativos, model.ativos, initialize = lambda model,a,b: sigma_usado.iloc[a,b])
    # model.s = pyo.Param(initialize = 2, mutable=True)
    # model.r = pyo.Var(within=pyo.NonNegativeReals)
    
    for i in range(3):
        setattr(model,f'obj{i}_mais',pyo.Var(domain=NonNegativeReals)) #bounds=(0,0.1),
        setattr(model,f'obj{i}_menos',pyo.Var(domain=NonNegativeReals))
    #-------------------------------------- FUNÇÕES

    #=============================
    # Função Objetivo
    #=============================

    def func_objetivo_1(model):
       
        return model.obj1_menos + model.obj2_menos
    model.objetivo = pyo.Objective(rule=func_objetivo_1, sense=pyo.minimize)

    #=============================
    # RESTRIÇÕES
    #=============================

    # # OBJ 1 2 3 - Rest 1 2 3
    # def retorno_restr_rule(model,diad):
    #     return sum(model.x[a]*model.retornos_ativos[dia,a] for a in model.ativos for dia in model.dias) + model.obj0_menos - model.obj0_mais >= model.retornos_ibov[diad]
    # model.retorno_restr = pyo.Constraint(model.dias,rule=retorno_restr_rule)

    def piotroski_restr_rule(model):
        return sum(model.x[a]*model.score_p[a] for a in model.ativos) + model.obj1_menos - model.obj1_mais >= 0.9
    model.piotroski_rest = pyo.Constraint(rule=piotroski_restr_rule)

    def mf_restr_rule(model):
        return sum(model.x[a]*model.score_m[a] for a in model.ativos) + model.obj2_menos - model.obj2_mais >= 0.9
    model.mf_restr = pyo.Constraint(rule=mf_restr_rule)

    #REstricao 1 x só ativa se y = 1
    def restr_vinculo_x_y(model, a):
        return model.x[a] <= model.y[a]
    model.const_restr_vinculo_x_y = pyo.Constraint(model.ativos, rule=restr_vinculo_x_y)

    #peso maximo por acao
    def rule_peso_maximo(model, a):
        # return model.x[a] <= 1/model.cardinalidade_valor
        return model.x[a] <= model.peso_maximo
    model.const_peso_maximo = pyo.Constraint(model.ativos, rule=rule_peso_maximo)

    # #peso minimo por acao
    # def rule_peso_minimo(model, a):
    #     return model.x[a] >= model.peso_minimo * model.y[a]  # se y=1, então x >= 0.05
    # model.const_peso_minimo = pyo.Constraint(model.ativos, rule=rule_peso_minimo)

    #Restrição 2 soma peso 1
    def soma_peso_1(model):
        return sum(model.x[a] for a in model.ativos) == 1
    model.const_soma_peso_1 = pyo.Constraint(rule=soma_peso_1)


    # def cardinalidade_min(model):
    #     return sum(
    #         model.y[a] for a in model.ativos
    #         ) >= model.cardinalidade_valor_min
    # model.const_cardinalidade_total_min = pyo.Constraint(rule=cardinalidade_min)

    def cardinalidade_max(model):
        return sum(
            model.y[a] for a in model.ativos
            ) <= model.cardinalidade_valor_max
    model.const_cardinalidade_total_max = pyo.Constraint(rule=cardinalidade_max)


    # NOTEBOOOK
    # opt = SolverFactory('cplex', executable='C:\\CPLEX_Studio2211\\cplex\\bin\\x64_win64\\cplex.exe')

    # PC
    opt = SolverFactory('cplex', executable='C:\\Program Files\\IBM\\ILOG\\CPLEX_Studio_Community222\\cplex\\bin\\x64_win64\\cplex.exe')
    res = opt.solve(model,tee=False)
    print("Situação SOLVER: ",res.solver.termination_condition)
    if (res.solver.status == SolverStatus.ok) and (res.solver.termination_condition == TerminationCondition.optimal): 
        melhor_pesos = {list(model.nome_ativos.data())[a]: pyo.value(model.x[a]) for a in model.ativos}
        pio = {sum(pyo.value(model.x[t])*pyo.value(model.score_p[t]) for t in model.ativos)}
        mf = {sum(pyo.value(model.x[t])*pyo.value(model.score_m[t]) for t in model.ativos)}
        carteiras_anuais[int(ano)] = {
            'pesos':  melhor_pesos,
            # 'sharpe_anual': s_lo*np.sqrt(252),
        }
        print(f"{ano} -> {melhor_pesos}")
        print(f"{int(ano)-1} Piotroski -> {pio}")
        print(f"Valores obj Menos: {pyo.value(model.obj1_menos), pyo.value(model.obj1_mais)}")
        print(f"{int(ano)-1} Magic Formula -> {mf}")
        print(f"Valores obj Menos: {pyo.value(model.obj2_menos), pyo.value(model.obj2_mais)}")
        print("+-="*30)
        if 'model' in locals():
            del model
            print('DELETADO DPS DO WHILE')
        else:
            print("Nao consta model")
            pass
    else:
        print("PROblema no ANO: ", ano)
        break




COMEÇANDO ANO NOVO:  2016
Nao consta model
ATIVOS:  Index(['ABEV3', 'ALOS3', 'ALPA4', 'ANIM3', 'AXIA3', 'AZZA3', 'B3SA3', 'BEEF3',
       'CMIG4', 'COGN3', 'CPFE3', 'CPLE3', 'CSAN3', 'CSMG3', 'CSNA3', 'CYRE3',
       'DIRR3', 'ECOR3', 'EGIE3', 'EMBJ3', 'ENEV3', 'EQTL3', 'EZTC3', 'FLRY3',
       'GGBR4', 'GOAU4', 'HYPE3', 'ISAE4', 'ITSA4', 'JHSF3', 'KLBN11', 'LREN3',
       'MBRF3', 'MGLU3', 'MOTV3', 'MRVE3', 'MULT3', 'NATU3', 'PETR3', 'PETR4',
       'POMO4', 'PRIO3', 'RADL3', 'RENT3', 'SBSP3', 'SLCE3', 'SMTO3', 'TAEE11',
       'TOTS3', 'UGPA3', 'USIM5', 'VALE3', 'VIVT3', 'WEGE3', 'YDUQ3'],
      dtype='str')
Score Atualizado:  2015
-----
2015 TAMANHOS: Retorno 55, Score PIO 55, Score MF 55
# ------ CRIAÇÃO DO MODELO
## Carteira criada para o ano:  2016
Situação SOLVER:  optimal
2016 -> {'ABEV3': 0.0, 'ALOS3': 0.0, 'ALPA4': 0.0, 'ANIM3': 0.0, 'AXIA3': 0.0, 'AZZA3': 0.0, 'B3SA3': 0.0, 'BEEF3': 0.0, 'CMIG4': 0.0, 'COGN3': 0.0, 'CPFE3': 0.0, 'CPLE3': 0.0, 'CSAN3': 0.0, 'CSMG3': 0.0, 'CSN

In [6]:
linhas = []
for an in anos:
    an = int(an.split('-')[0])
    print(an)
    for k, v in carteiras_anuais[an]['pesos'].items():
        if round(v, 4)>0.000 and v <= vb_peso_maximo*1.10:
            linhas.append({'ano': an, 'ativo': k, 'peso': v})
df_portfolios = pd.DataFrame(linhas)

# # visões instantâneas:
# df_portfolios[df_portfolios['ano'] == 2016].sort_values('peso', ascending=False)  # uma carteira
# df_portfolios.pivot(index='ativo', columns='ano', values='peso')                  # matriz ativo × ano
# df_portfolios.groupby('ativo')['ano'].count().sort_values(ascending=False)  


2016
2017
2018
2019
2020
2021
2022
2023
2024
2025
2026


In [7]:
df_portfolios[df_portfolios['ano'] == 2026].sort_values('peso', ascending=False)

,ano,ativo,peso
71,2026,ABEV3,0.15
72,2026,CEAB3,0.15
73,2026,CVCB3,0.15
74,2026,JHSF3,0.15
76,2026,SAUD3,0.15
77,2026,TIMS3,0.15
75,2026,LREN3,0.10


In [8]:
df_portfolios.to_csv('../carteiras/goal_programming/carteiras_goal_programming.csv')


# Linear Sharpe - Magic Formula

In [9]:
y = []
carteiras_anuais = {}
melhor_pesos = None
historico = []
carteiras_criadas = []
for a in anos:
    y.append(a.split('-')[0])
print("=-"*48)
print("Anos totais: ",y)
print("=-"*48)
lista_ativos_finais = {}
for an in anos:
    ano = an.split("-")[0]
    ano_int = int(ano)	
    print("COMEÇANDO ANO NOVO: ",ano)
    #deletar modelo

    if 'model' in locals():
        del model
        print('Modelo Antigo deletado \n Iniciando Novo')
    else:
        print("Nao consta model")
        pass
    try:
        # retorno_usado = pd.DataFrame(eval(df_usado))
        df_usado = dfs[str(ano_int)]

        retorno_usado_r = df_usado.copy()
        # print(retorno_usado_r)
        score_todos_anos = pd.read_excel(f'../../../score-raspagem/magic_formula/resultado/{ano_int-1}.xlsx')
        score_usado_p = score_todos_anos[['Unnamed: 0', 'magic_formula']].set_index('Unnamed: 0').T
        
        ativos = retorno_usado_r.columns.intersection(score_usado_p.columns)
        # print("ATIVOS: ",ativos)
        retorno_usado = retorno_usado_r[ativos]
        retorno_usado = (retorno_usado - retorno_usado.min())/(retorno_usado.max() - retorno_usado.min())
        # print(retorno_usado)
        df_cov = retorno_usado.cov()
        # print(df_cov)

        score = score_usado_p[ativos]
        # print(score)
        # COV
        # print(df_cov)
        # Lista ativos
        lista_ativos_finais[ano] = retorno_usado.columns.tolist()
      
    except Exception as e:
            print("=========ERRRRRRRRRRRROR")
            print(e)    

    print("# ------ CRIAÇÃO DO MODELO")
    print("## Carteira criada para o ano: ",ano_int)
    print("## UTILIZANDO DADOS DE RETORNO DE: ",ano)

    #  --------- MODELO
   

    #  --------- MODELO
    model = pyo.ConcreteModel()
    #---------VARIÁVEIS-----------
    model.nome_ativos = pyo.Set(initialize = retorno_usado.columns)
    model.ativos = pyo.RangeSet(0, len(retorno_usado.columns)-1)
    model.dias = pyo.RangeSet(0, len(retorno_usado)-1)
    model.retornos_ativos = pyo.Param(model.dias, model.ativos, initialize=lambda model,dia, ativo: retorno_usado.iloc[dia, ativo])    
    model.sigma = pyo.Param(model.ativos, model.ativos, initialize=lambda model,a,b: df_cov.iloc[a,b])
    # model.sigma_min = pyo.Param(initialize=0, mutable=True)
    # model.sigma_max = pyo.Param(initialize=1, mutable=True)
    model.score = pyo.Param(model.ativos, initialize=lambda model,a: score.iloc[0,a])
    model.theta = pyo.Param(initialize=vb_theta)
    model.PESOMAX = pyo.Param(initialize = vb_peso_maximo)
    # ------------------------------

    #linearizar Sharpe
    # z:= kx
    model.z = pyo.Var(model.ativos, domain=pyo.NonNegativeReals)
    model.k = pyo.Var(domain=pyo.NonNegativeReals)
    model.y = pyo.Var(model.ativos, domain=pyo.Binary)
    model.w = pyo.Var(model.ativos, domain=pyo.NonNegativeReals)
    model.KMAX = pyo.Param(initialize=100)

    # Com a problemática de cardinalidade y e bilinearidade, resolve-se
    # PESO
    # antes era model.x[a] <= model.pesomax * model.y[a]
    #  z=kx  -> model.z[a] <= model.pesomax * modely[a] * model.k[a]
    # CARDINALIDADE
    # era sum(model.y[a] ...) >= e <=
    # é feita a tranformação de w <= k * y

    # Com y=0 
    def res_y_zero(model,a):
            return model.w[a] <= model.KMAX * model.y[a]
    model.y_zero = pyo.Constraint(model.ativos, rule=res_y_zero)

    #--------------------------------------------------------------------------
    # com y=1
    #  segundo teto, quando y=1 ; w = k , pelas restriçoes >= e <=
    def res_y_1_teto2(model,a):
            return model.w[a] <= model.k 
    model.res_teto2 = pyo.Constraint(model.ativos, rule=res_y_1_teto2)

    # Limite inf 1
    def limite_1(model,a):
            return model.w[a] >= model.k - model.KMAX*(1-model.y[a])
    model.r_limite1 = pyo.Constraint(model.ativos, rule=limite_1)
    #--------------------------------------------------------------------------
    #  PESOS MIN MAX
    # def pesomin(model,a):
    #         return model.z[a] >= 0.02 * model.w[a]
    # model.r_pesomin = pyo.Constraint(model.ativos, rule=pesomin)

    def pesomax(model,a):
            return model.z[a] <= model.PESOMAX * model.w[a]
    model.r_pesomax = pyo.Constraint(model.ativos, rule=pesomax)

    # #  cardinalidade
    # def card_min(model):
    #         return sum(model.y[a] for a in model.ativos) >= vb_cardinalidade_min
    # model.r_cardmin = pyo.Constraint(rule=card_min)
    def card_max(model):
            return sum(model.y[a] for a in model.ativos) <= vb_cardinalidade_max
    model.r_card_max = pyo.Constraint(rule=card_max)

    # Restrições Sharpe
    def restricao_z(model):
            return sum(model.z[a] for a in model.ativos) == model.k
    model.r_restricao_z = pyo.Constraint(rule=restricao_z)

    def restricao_z_retorno(model):
            return sum(model.sigma[a,b]*model.z[a]*model.z[b] for a in model.ativos for b in model.ativos)  <= 1
    model.r_res_z_retorno = pyo.Constraint(rule=restricao_z_retorno)

    ## OBJETIVOS:  1 para pegar z min, 2 para z max, 3 para normalização
    def obj_1(model):
        n_dias = len(model.dias)
        var = sum(model.z[a] * model.retornos_ativos[d,a] for a in model.ativos for d in model.dias) / n_dias
        score = sum(model.z[a] * model.score[a] for a in model.ativos)
        return model.theta*var + (1-model.theta)*score
    model.obj1 = pyo.Objective(rule=obj_1, sense=pyo.maximize)

#     NOTEBOOOK
    # opt = SolverFactory('cplex', executable='C:\\CPLEX_Studio2211\\cplex\\bin\\x64_win64\\cplex.exe')

    # ------------------- solver 1
    opt = SolverFactory('cplex', executable='C:\\Program Files\\IBM\\ILOG\\CPLEX_Studio_Community222\\cplex\\bin\\x64_win64\\cplex.exe')
    # res = opt.solve(model,tee=False,symbolic_solver_labels=True)
    res = opt.solve(model,tee=False)
    # log_infeasible_constraints(model, log_expression=True, log_variables=True)
    # logging.basicConfig(filename='example.log', encoding='utf-8', level=logging.INFO)

    # model.write('modelo_debug.lp', io_options={'symbolic_solver_labels': True})
    print(f"Valor Objetivo: {model.obj1()}")
    melhor_pesos = {list(model.nome_ativos.data())[a]:(model.z[a].value/model.k.value) for a in model.ativos}
    carteiras_anuais[int(ano_int)] = {
    	'pesos':  melhor_pesos,
    	# 'sharpe_anual': s_lo*np.sqrt(252),
    }
    print(f"{ano_int} -> {melhor_pesos}")
    print("+-="*30)
    if 'model' in locals():
    	del model
    	print('DELETADO DPS DO WHILE')
    else:
    	print("Nao consta model")
    	pass



=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-
Anos totais:  ['2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025', '2026']
=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-
COMEÇANDO ANO NOVO:  2016
Nao consta model
# ------ CRIAÇÃO DO MODELO
## Carteira criada para o ano:  2016
## UTILIZANDO DADOS DE RETORNO DE:  2016
Valor Objetivo: 7.30299195138475
2016 -> {'ABEV3': 0.034205417125502124, 'ALOS3': 0.0, 'ALPA4': 0.059646825869081495, 'ANIM3': 0.0, 'AXIA3': 0.0, 'AZZA3': 0.0, 'B3SA3': 0.0, 'BBAS3': 0.0, 'BBDC3': 0.0, 'BBDC4': 0.0, 'BBSE3': 0.08234695042890466, 'BEEF3': 0.0, 'CMIG4': 0.0, 'COGN3': 0.0, 'CPFE3': 0.0, 'CPLE3': 0.0, 'CSAN3': 0.0, 'CSMG3': 0.0, 'CSNA3': 0.0, 'CVCB3': 0.14999997612903437, 'CYRE3': 0.0, 'DIRR3': 0.0, 'ECOR3': 0.0, 'EGIE3': 0.0, 'EMBJ3': 0.09901300431542519, 'ENEV3': 0.0652386291773572, 'EQTL3': 0.0, 'EZTC3': 0.0, 'FLRY3': 0.0223417322329

In [10]:
linhas = []
for an in anos:
    an = int(an.split('-')[0])
    print(an)
    for k, v in carteiras_anuais[an]['pesos'].items():
        # if v >= vb_peso_minimo and v <= vb_peso_maximo:
        if round(v, 4)>0.000 and v <= vb_peso_maximo*1.10:
            linhas.append({'ano': an, 'ativo': k, 'peso': v})
df_portfolios = pd.DataFrame(linhas)



2016
2017
2018
2019
2020
2021
2022
2023
2024
2025
2026


In [11]:
print(f"THETA : {vb_theta}")
print(f"1 - THETA : {1 - vb_theta}")
df_portfolios[df_portfolios['ano'] == 2026].sort_values('peso', ascending=False)

THETA : 0.1
1 - THETA : 0.9


,ano,ativo,peso
138,2026,CVCB3,0.150000
143,2026,SAUD3,0.150000
139,2026,ENEV3,0.140796
134,2026,BBSE3,0.127582
140,2026,GMAT3,0.115873
135,2026,CBAV3,0.067718
142,2026,PRIO3,0.063348
141,2026,PETR4,0.055610
136,2026,CEAB3,0.051553
145,2026,WEGE3,0.029350


In [12]:
df_portfolios.to_csv('../carteiras/linear_sharpe_mf/carteiras_sharpe_linear_magic_formula.csv')


# Linear Sharpe - Piotroski

In [13]:
y = []
carteiras_anuais = {}
melhor_pesos = None
historico = []
carteiras_criadas = []
for a in anos:
    y.append(a.split('-')[0])
print("=-"*48)
print("Anos totais: ",y)
print("=-"*48)
lista_ativos_finais = {}
for an in anos:
    ano = an.split("-")[0]
    ano_int = int(ano)	
    print("COMEÇANDO ANO NOVO: ",ano)
    #deletar modelo
    if 'model' in locals():
        del model
        print('Modelo Antigo deletado \n Iniciando Novo')
    else:
        print("Nao consta model")
        pass
    try:
        # retorno_usado = pd.DataFrame(eval(df_usado))
        df_usado = dfs[str(ano_int)]
        retorno_usado = df_usado.copy()

        score_todos_anos = pd.read_excel(f'../../../score-raspagem/piotroski/resultado/{ano_int-1}.xlsx')
        score_usado_p = score_todos_anos[['Unnamed: 0', 'p_norm']].set_index('Unnamed: 0').T
        
        ativos = retorno_usado.columns.intersection(score_usado_p.columns)
        retorno_usado = retorno_usado[ativos]
        retorno_usado = (retorno_usado-retorno_usado.min())/(retorno_usado.max()-retorno_usado.min())
        score = score_usado_p[ativos]

        # COV
        df_cov = retorno_usado.cov()

        # Lista ativos
        lista_ativos_finais[ano] = retorno_usado.columns.tolist()
      
    except Exception as e:
            print("=========ERRRRRRRRRRRROR")
            print(e)    

    print("# ------ CRIAÇÃO DO MODELO")
    print("## Carteira criada para o ano: ",ano_int+1)
    print("## UTILIZANDO DADOS DE RETORNO DE: ",ano)

    #  --------- MODELO
   

    #  --------- MODELO
    model = pyo.ConcreteModel()
    #---------VARIÁVEIS-----------
    model.nome_ativos = pyo.Set(initialize = retorno_usado.columns)
    model.ativos = pyo.RangeSet(0, len(retorno_usado.columns)-1)
    model.dias = pyo.RangeSet(0, len(retorno_usado)-1)
    model.retornos_ativos = pyo.Param(model.dias, model.ativos, initialize=lambda model,dia, ativo: retorno_usado.iloc[dia, ativo])    
    model.sigma = pyo.Param(model.ativos, model.ativos, initialize=lambda model,a,b: df_cov.iloc[a,b])
    # model.sigma_min = pyo.Param(initialize=0, mutable=True)
    # model.sigma_max = pyo.Param(initialize=1, mutable=True)
    model.score = pyo.Param(model.ativos, initialize=lambda model,a: score.iloc[0,a])
    model.theta = pyo.Param(initialize=vb_theta)
    model.PESOMAX = pyo.Param(initialize = vb_peso_maximo)

    # ------------------------------

    #linearizar Sharpe
    # z:= kx
    model.z = pyo.Var(model.ativos, domain=pyo.NonNegativeReals)
    model.k = pyo.Var(domain=pyo.NonNegativeReals)
    model.y = pyo.Var(model.ativos, domain=pyo.Binary)
    model.w = pyo.Var(model.ativos, domain=pyo.NonNegativeReals)
    model.KMAX = pyo.Param(initialize=100)

    # Com a problemática de cardinalidade y e bilinearidade, resolve-se
    # PESO
    # antes era model.x[a] <= model.pesomax * model.y[a]
    #  z=kx  -> model.z[a] <= model.pesomax * modely[a] * model.k[a]
    # CARDINALIDADE
    # era sum(model.y[a] ...) >= e <=
    # é feita a tranformação de w <= k * y

    # Com y=0 
    def res_y_zero(model,a):
            return model.w[a] <= model.KMAX * model.y[a]
    model.y_zero = pyo.Constraint(model.ativos, rule=res_y_zero)

    #--------------------------------------------------------------------------
    # com y=1
    #  segundo teto, quando y=1 ; w = k , pelas restriçoes >= e <=
    def res_y_1_teto2(model,a):
            return model.w[a] <= model.k 
    model.res_teto2 = pyo.Constraint(model.ativos, rule=res_y_1_teto2)

    # Limite inf 1
    def limite_1(model,a):
            return model.w[a] >= model.k - model.KMAX*(1-model.y[a])
    model.r_limite1 = pyo.Constraint(model.ativos, rule=limite_1)
    #--------------------------------------------------------------------------
    # #  PESOS MIN MAX
    # def pesomin(model,a):
    #         return model.z[a] >= 0.02 * model.w[a]
    # model.r_pesomin = pyo.Constraint(model.ativos, rule=pesomin)

    def pesomax(model,a):
            return model.z[a] <=model.PESOMAX* model.w[a]
    model.r_pesomax = pyo.Constraint(model.ativos, rule=pesomax)

    # #  cardinalidade
    # def card_min(model):
    #         return sum(model.y[a] for a in model.ativos) >= vb_cardinalidade_min
    # model.r_cardmin = pyo.Constraint(rule=card_min)
    def card_max(model):
            return sum(model.y[a] for a in model.ativos) <= vb_cardinalidade_max
    model.r_card_max = pyo.Constraint(rule=card_max)

    # Restrições Sharpe
    def restricao_z(model):
            return sum(model.z[a] for a in model.ativos) == model.k
    model.r_restricao_z = pyo.Constraint(rule=restricao_z)

    def restricao_z_retorno(model):
            return sum(model.sigma[a,b]*model.z[a]*model.z[b] for a in model.ativos for b in model.ativos)  <= 1
    model.r_res_z_retorno = pyo.Constraint(rule=restricao_z_retorno)

    ## OBJETIVOS:  1 para pegar z min, 2 para z max, 3 para normalização
    def obj_1(model):
        n_dias = len(model.dias)
        var = sum(model.z[a] * model.retornos_ativos[d,a] for a in model.ativos for d in model.dias) / n_dias
        score = sum(model.z[a] * model.score[a] for a in model.ativos)
        return model.theta*var + (1-model.theta)*score
    model.obj1 = pyo.Objective(rule=obj_1, sense=pyo.maximize)


    # NOTEBOOOK
    # opt = SolverFactory('cplex', executable='C:\\CPLEX_Studio2211\\cplex\\bin\\x64_win64\\cplex.exe')

    # ------------------- solver 1
    opt = SolverFactory('cplex', executable='C:\\Program Files\\IBM\\ILOG\\CPLEX_Studio_Community222\\cplex\\bin\\x64_win64\\cplex.exe')
    res = opt.solve(model,tee=False)
#     model.write('modelo_debug.lp', io_options={'symbolic_solver_labels': True})
   
    melhor_pesos = {list(model.nome_ativos.data())[a]:(model.z[a].value/model.k.value) for a in model.ativos}
    carteiras_anuais[int(ano_int)] = {
    	'pesos':  melhor_pesos,
    	# 'sharpe_anual': s_lo*np.sqrt(252),
    }
    print(f"{ano_int} -> {melhor_pesos}")
    print("+-="*30)
    if 'model' in locals():
    	del model
    	print('DELETADO DPS DO WHILE')
    else:
    	print("Nao consta model")
    	pass



=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-
Anos totais:  ['2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025', '2026']
=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-
COMEÇANDO ANO NOVO:  2016
Nao consta model
# ------ CRIAÇÃO DO MODELO
## Carteira criada para o ano:  2017
## UTILIZANDO DADOS DE RETORNO DE:  2016
2016 -> {'ABEV3': 0.0, 'ALOS3': 0.0, 'ALPA4': 0.06558028092486395, 'ANIM3': 0.0, 'AXIA3': 0.0, 'AZZA3': 0.0, 'B3SA3': 0.0, 'BEEF3': 0.0, 'CMIG4': 0.0, 'COGN3': 0.00747848388935855, 'CPFE3': 0.0, 'CPLE3': 0.0, 'CSAN3': 0.06809281721423607, 'CSMG3': 0.0, 'CSNA3': 0.0, 'CYRE3': 0.029903617153942876, 'DIRR3': 0.0, 'ECOR3': 0.0, 'EGIE3': 0.0, 'EMBJ3': 0.0, 'ENEV3': 0.10120965305895686, 'EQTL3': 1.2554631010674103e-09, 'EZTC3': 0.0, 'FLRY3': 0.12430150255365045, 'GGBR4': 0.0, 'GOAU4': 0.0, 'HYPE3': 0.0, 'ISAE4': 0.0, 'ITSA4': 0.0, 'JHSF3': 0.0464818420462

In [14]:
linhas = []
for an in anos:
    an = int(an.split('-')[0])
    print(an)
    for k, v in carteiras_anuais[an]['pesos'].items():
        #if v >= vb_peso_minimo and v <= vb_peso_maximo:
        if round(v, 4)>0.0001 and v <= vb_peso_maximo*1.10:
            linhas.append({'ano': an, 'ativo': k, 'peso': v})

df_portfolios = pd.DataFrame(linhas)

# # visões instantâneas:
# df_portfolios[df_portfolios['ano'] == 2016].sort_values('peso', ascending=False)  # uma carteira
# df_portfolios.pivot(index='ativo', columns='ano', values='peso')                  # matriz ativo × ano
# df_portfolios.groupby('ativo')['ano'].count().sort_values(ascending=False)  


2016
2017
2018
2019
2020
2021
2022
2023
2024
2025
2026


In [15]:
df_portfolios[df_portfolios['ano'] == 2020]

,ano,ativo,peso
59,2020,BEEF3,0.100696
60,2020,CSMG3,0.150000
61,2020,CSNA3,0.149304
62,2020,MOTV3,0.150000
63,2020,SBSP3,0.150000
64,2020,TIMS3,0.150000
65,2020,WEGE3,0.150000


In [16]:
df_portfolios.to_csv('../carteiras/linear_sharpe_piotroski/carteiras_sharpe_linear_piotroski.csv')


# Mínima Variância - Magic Formula

In [17]:
y = []

carteiras_anuais = {}

melhor_pesos = None
historico = []
carteiras_criadas = []
for a in anos:
    y.append(a.split('-')[0])

print("=-"*48)
print("Anos totais: ",y)
print("=-"*48)

for an in anos:
    ano = an.split("-")[0]
    print("COMEÇANDO ANO NOVO: ",ano)
    #deletar modelo
    if 'model' in locals():
        del model
        print('Modelo Antigo deletado \n Iniciando Novo')
    else:
        print("Nao consta model")
        pass


    ano_um = int(ano)
    # df_usado = f'df_ativos_{ano_um}'
    # retorno_usado = pd.DataFrame(eval(df_usado))
    df_usado = dfs[str(ano_um)]
    retorno_usado = df_usado.copy()

    score_todos_anos = pd.read_excel(f'../../../score-raspagem/magic_formula/resultado/{ano_um-1}.xlsx')
    score_usado_p = score_todos_anos[['Unnamed: 0', 'magic_formula']].set_index('Unnamed: 0').T
    
    ativos = retorno_usado.columns.intersection(score_usado_p.columns)
    retorno_usado = retorno_usado[ativos]
    retorno_usado = (retorno_usado - retorno_usado.min())/(retorno_usado.max() - retorno_usado.min())

    score = score_usado_p[ativos]

    print("Retornos atualizados")

    sigma_usado = retorno_usado.cov()
    print("Sigmas Atualizados")
    
    print("-----")
    
    print("# ------ CRIAÇÃO DO MODELO")
    print("## Carteira criada para o ano: ",ano_um)

    print("## UTILIZANDO SCORE DO ANO DE: ",ano_um-1)
    print("## UTILIZANDO DADOS DE RETORNO DE: ",str(ano_um))
    # print("## UTILIZANDO EXCESSO DO ANO DE: ",ano_um)
    print("## UTILIZANDO SIGMAS DO ANO DE: ",ano_um)

    ## NORMALIZAÇÃO DE ESCALA F.O
    # valores típicos de cada termo, pra deixar os dois na mesma escala
    # escala_var = float(np.diag(sigma_usado.values).mean())   # variância média de um ativo
    escala_var = float(sigma_usado.values.mean()) # ESCALA VARIÂNCIA MÉDIA DE TODOS OS ATIVOS
    escala_score = float(score.iloc[0].mean())               # score médio dos ativos
    print(f"Ano {ano} | escala var: {escala_var:.5f} | escala score: {escala_score:.4f}")

    model = pyo.ConcreteModel()

    #---------VARIÁVEIS-----------
    model.nome_ativos = pyo.Set(initialize = retorno_usado.columns.tolist())
    model.ativos = pyo.RangeSet(0, len(retorno_usado.columns.tolist())-1)
    model.dias = pyo.RangeSet(0, len(retorno_usado)-1)
    model.retornos_ativos = pyo.Param(model.dias, model.ativos, initialize=lambda model,dia, ativo: retorno_usado.iloc[dia, ativo])    
    model.theta = pyo.Param(initialize=vb_theta)
    model.score = pyo.Param(model.ativos, initialize=lambda model,a: score.iloc[0,a])
    model.cardinalidade_valor_max = pyo.Param(initialize=vb_cardinalidade_max)
    # model.cardinalidade_valor_min = pyo.Param(initialize=vb_cardinalidade_min)
    model.peso_maximo = pyo.Param(initialize=vb_peso_maximo)
    # model.peso_minimo = pyo.Param(initialize=vb_peso_minimo)
    model.x = pyo.Var(model.ativos,bounds=(0,1),domain=pyo.NonNegativeReals)
    model.y = pyo.Var(model.ativos, within=pyo.Binary)
    # model.excesso = pyo.Param( model.ativos , initialize = lambda model,a: excesso_usado.mean().iloc[a])
    model.sigma = pyo.Param(model.ativos, model.ativos, initialize = lambda model,a,b: sigma_usado.iloc[a,b])
    # model.s = pyo.Param(initialize = 2, mutable=True)
    # model.r = pyo.Var(within=pyo.NonNegativeReals)

    #-------------------------------------- FUNÇÕES

    #=============================
    # Função Objetivo
    #=============================
    

    def func_objetivo_1(model):
        min_var = sum(model.x[a]*model.x[b]*model.sigma[a,b] for a in model.ativos for b in model.ativos)
        score = sum(model.x[a] * model.score[a] for a in model.ativos)
        return model.theta*(min_var/escala_var) - (1-model.theta)*(score/escala_score)
    model.obj1 = pyo.Objective(rule=func_objetivo_1, sense=pyo.minimize)
    #=============================
    # RESTRIÇÕES
    #=============================
    #REstricao 1 x só ativa se y = 1
    def restr_vinculo_x_y(model, a):
        return model.x[a] <= model.y[a]
    model.const_restr_vinculo_x_y = pyo.Constraint(model.ativos, rule=restr_vinculo_x_y)

    #peso maximo por acao
    def rule_peso_maximo(model, a):
        # return model.x[a] <= 1/model.cardinalidade_valor
        return model.x[a] <= model.peso_maximo * model.y[a]  # se y=1, então x <= 0.20
    model.const_peso_maximo = pyo.Constraint(model.ativos, rule=rule_peso_maximo)

    # #peso minimo por acao
    # def rule_peso_minimo(model, a):
    #     return model.x[a] >= model.peso_minimo * model.y[a]  # se y=1, então x >= 0.05
    # model.const_peso_minimo = pyo.Constraint(model.ativos, rule=rule_peso_minimo)

    #Restrição 2 soma peso 1
    def soma_peso_1(model):
        return sum(model.x[a] for a in model.ativos) == 1
    model.const_soma_peso_1 = pyo.Constraint(rule=soma_peso_1)

    # def cardinalidade_min(model):
    #     return sum(
    #         model.y[a] for a in model.ativos
    #         ) >= model.cardinalidade_valor_min
    # model.const_cardinalidade_total_min = pyo.Constraint(rule=cardinalidade_min)

    def cardinalidade_max(model):
        return sum(
            model.y[a] for a in model.ativos
            ) <= model.cardinalidade_valor_max
    model.const_cardinalidade_total_max = pyo.Constraint(rule=cardinalidade_max)


    # NOTEBOOOK
    # opt = SolverFactory('cplex', executable='C:\\CPLEX_Studio2211\\cplex\\bin\\x64_win64\\cplex.exe')

    # PC
    opt = SolverFactory('cplex', executable='C:\\Program Files\\IBM\\ILOG\\CPLEX_Studio_Community222\\cplex\\bin\\x64_win64\\cplex.exe')
    res = opt.solve(model,tee=False)
    if (res.solver.status == SolverStatus.ok) and (res.solver.termination_condition == TerminationCondition.optimal): 

        melhor_pesos = {list(model.nome_ativos.data())[a]: pyo.value(model.x[a]) for a in model.ativos}
        carteiras_anuais[int(ano_um)] = {
            'pesos':  melhor_pesos,
            # 'sharpe_anual': s_lo*np.sqrt(252),
        }
        print(f"{ano_um} -> {melhor_pesos}")
        print("+-="*30)
        if 'model' in locals():
            del model
            print('DELETADO DPS DO WHILE')
        else:
            print("Nao consta model")
            pass
    else:
        print("PROblema no ANO: ", ano)
        print(res.solver.termination_condition)
        break




=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-
Anos totais:  ['2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025', '2026']
=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-
COMEÇANDO ANO NOVO:  2016
Nao consta model
Retornos atualizados
Sigmas Atualizados
-----
# ------ CRIAÇÃO DO MODELO
## Carteira criada para o ano:  2016
## UTILIZANDO SCORE DO ANO DE:  2015
## UTILIZANDO DADOS DE RETORNO DE:  2016
## UTILIZANDO SIGMAS DO ANO DE:  2016
Ano 2016 | escala var: 0.01154 | escala score: 0.4128
2016 -> {'ABEV3': 0.14999999999999997, 'ALOS3': 0.0, 'ALPA4': 0.08390720609415132, 'ANIM3': 0.0, 'AXIA3': 0.0, 'AZZA3': 0.029840478862898513, 'B3SA3': 0.0, 'BBAS3': 0.0, 'BBDC3': 0.0, 'BBDC4': 0.0, 'BBSE3': 0.15, 'BEEF3': 0.0, 'CMIG4': 0.0, 'COGN3': 0.0, 'CPFE3': 0.0, 'CPLE3': 0.0, 'CSAN3': 0.0, 'CSMG3': 0.0, 'CSNA3': 0.0, 'CVCB3': 0.15000000000000002, 'CYRE3': 0.0, 'DIRR3': 0.

In [18]:
linhas = []
for an in anos:
    an = int(an.split('-')[0])
    print(an)
    for k, v in carteiras_anuais[an]['pesos'].items():
        if round(v, 4)>0.000 and v <= vb_peso_maximo*1.10:
            linhas.append({'ano': an, 'ativo': k, 'peso': v})

df_portfolios = pd.DataFrame(linhas)

# # visões instantâneas:
# df_portfolios[df_portfolios['ano'] == 2016].sort_values('peso', ascending=False)  # uma carteira
# df_portfolios.pivot(index='ativo', columns='ano', values='peso')                  # matriz ativo × ano
# df_portfolios.groupby('ativo')['ano'].count().sort_values(ascending=False)  


2016
2017
2018
2019
2020
2021
2022
2023
2024
2025
2026


In [19]:
print(f"THETA = {vb_theta}")
df_portfolios[df_portfolios['ano'] == 2020].sort_values('peso', ascending=False)

THETA = 0.1


,ano,ativo,peso
37,2020,BBSE3,0.15
40,2020,SAUD3,0.15
35,2020,ABEV3,0.15
39,2020,LREN3,0.15
38,2020,CEAB3,0.15
41,2020,SBFG3,0.15
36,2020,AZZA3,0.10


In [20]:
df_portfolios.to_csv('../carteiras/min_var_mf/carteiras_minvar_magic_formula.csv')


# Mínima Variância - Piotroski

In [21]:
y = []

carteiras_anuais = {}

melhor_pesos = None
historico = []
carteiras_criadas = []
for a in anos:
    y.append(a.split('-')[0])

print("=-"*48)
print("Anos totais: ",y)
print("=-"*48)

for an in anos:
    ano = an.split("-")[0]
    print("COMEÇANDO ANO NOVO: ",ano)
    #deletar modelo
    if 'model' in locals():
        del model
        print('Modelo Antigo deletado \n Iniciando Novo')
    else:
        print("Nao consta model")
        pass


    ano_um = int(ano)
    # df_usado = f'df_ativos_{ano_um}'
    # retorno_usado = pd.DataFrame(eval(df_usado))
    df_usado = dfs[str(ano_um)]
    retorno_usado = df_usado.copy()

    score_todos_anos = pd.read_excel(f'../../../score-raspagem/piotroski/resultado/{ano_um-1}.xlsx')
    score_usado_p = score_todos_anos[['Unnamed: 0', 'p_norm']].set_index('Unnamed: 0').T
    
    ativos = retorno_usado.columns.intersection(score_usado_p.columns)
    retorno_usado = retorno_usado[ativos]
    retorno_usado = (retorno_usado - retorno_usado.min())/(retorno_usado.max() - retorno_usado.min())

    score = score_usado_p[ativos]

    print("Retornos atualizados")

    sigma_usado = retorno_usado.cov()
    print("Sigmas Atualizados")
    
    print("-----")
    
    print("# ------ CRIAÇÃO DO MODELO")
    print("## Carteira criada para o ano: ",int(ano_um))

    print("## UTILIZANDO SCORE DO ANO DE: ",ano_um-1)
    print("## UTILIZANDO DADOS DE RETORNO DE: ",str(ano_um))
    # print("## UTILIZANDO EXCESSO DO ANO DE: ",ano_um)
    print("## UTILIZANDO SIGMAS DO ANO DE: ",ano_um)

    ## NORMALIZAÇÃO DE ESCALA F.O
    # valores típicos de cada termo, pra deixar os dois na mesma escala
    # escala_var = float(np.diag(sigma_usado.values).mean())   # variância média de um ativo
    escala_var = float(sigma_usado.values.mean()) # ESCALA VARIÂNCIA MÉDIA DE TODOS OS ATIVOS
    escala_score = float(score.iloc[0].mean())               # score médio dos ativos
    print(f"Ano {ano} | escala var: {escala_var:.5f} | escala score: {escala_score:.4f}")

    model = pyo.ConcreteModel()

    #---------VARIÁVEIS-----------
    model.nome_ativos = pyo.Set(initialize = retorno_usado.columns.tolist())
    model.ativos = pyo.RangeSet(0, len(retorno_usado.columns.tolist())-1)
    model.dias = pyo.RangeSet(0, len(retorno_usado)-1)
    model.retornos_ativos = pyo.Param(model.dias, model.ativos, initialize=lambda model,dia, ativo: retorno_usado.iloc[dia, ativo])    
    model.theta = pyo.Param(initialize=vb_theta)
    model.score = pyo.Param(model.ativos, initialize=lambda model,a: score.iloc[0,a])
    model.cardinalidade_valor_max = pyo.Param(initialize=vb_cardinalidade_max)
    # model.cardinalidade_valor_min = pyo.Param(initialize=vb_cardinalidade_min)
    model.peso_maximo = pyo.Param(initialize=vb_peso_maximo)
    # model.peso_minimo = pyo.Param(initialize=vb_peso_minimo)
    model.x = pyo.Var(model.ativos,bounds=(0,1),domain=pyo.NonNegativeReals)
    model.y = pyo.Var(model.ativos, within=pyo.Binary)
    # model.excesso = pyo.Param( model.ativos , initialize = lambda model,a: excesso_usado.mean().iloc[a])
    model.sigma = pyo.Param(model.ativos, model.ativos, initialize = lambda model,a,b: sigma_usado.iloc[a,b])
    # model.s = pyo.Param(initialize = 2, mutable=True)
    # model.r = pyo.Var(within=pyo.NonNegativeReals)
    
    #-------------------------------------- FUNÇÕES

    #=============================
    # Função Objetivo
    #=============================

    def func_objetivo_1(model):
        min_var = sum(model.x[a]*model.x[b]*model.sigma[a,b] for a in model.ativos for b in model.ativos)
        score = sum(model.x[a] * model.score[a] for a in model.ativos)
        return model.theta*(min_var/escala_var) - (1-model.theta)*(score/escala_score)
    model.obj1 = pyo.Objective(rule=func_objetivo_1, sense=pyo.minimize)
    
    #=============================
    # RESTRIÇÕES
    #=============================
    #REstricao 1 x só ativa se y = 1
    def restr_vinculo_x_y(model, a):
        return model.x[a] <= model.y[a]
    model.const_restr_vinculo_x_y = pyo.Constraint(model.ativos, rule=restr_vinculo_x_y)

    #peso maximo por acao
    def rule_peso_maximo(model, a):
        # return model.x[a] <= 1/model.cardinalidade_valor
        return model.x[a] <= model.peso_maximo * model.y[a]  # se y=1, então x <= 0.20
    model.const_peso_maximo = pyo.Constraint(model.ativos, rule=rule_peso_maximo)

    # #peso minimo por acao
    # def rule_peso_minimo(model, a):
    #     return model.x[a] >= model.peso_minimo * model.y[a]  # se y=1, então x >= 0.05
    # model.const_peso_minimo = pyo.Constraint(model.ativos, rule=rule_peso_minimo)

    #Restrição 2 soma peso 1
    def soma_peso_1(model):
        return sum(model.x[a] for a in model.ativos) == 1
    model.const_soma_peso_1 = pyo.Constraint(rule=soma_peso_1)

    # def cardinalidade_min(model):
    #     return sum(
    #         model.y[a] for a in model.ativos
    #         ) >= model.cardinalidade_valor_min
    # model.const_cardinalidade_total_min = pyo.Constraint(rule=cardinalidade_min)

    def cardinalidade_max(model):
        return sum(
            model.y[a] for a in model.ativos
            ) <= model.cardinalidade_valor_max
    model.const_cardinalidade_total_max = pyo.Constraint(rule=cardinalidade_max)


    # NOTEBOOOK
    # opt = SolverFactory('cplex', executable='C:\\CPLEX_Studio2211\\cplex\\bin\\x64_win64\\cplex.exe')

    # PC
    opt = SolverFactory('cplex', executable='C:\\Program Files\\IBM\\ILOG\\CPLEX_Studio_Community222\\cplex\\bin\\x64_win64\\cplex.exe')
    res = opt.solve(model,tee=False)
    if (res.solver.status == SolverStatus.ok) and (res.solver.termination_condition == TerminationCondition.optimal): 
        melhor_pesos = {list(model.nome_ativos.data())[a]: pyo.value(model.x[a]) for a in model.ativos}
        carteiras_anuais[int(ano_um)] = {
            'pesos':  melhor_pesos,
            # 'sharpe_anual': s_lo*np.sqrt(252),
        }
        print(f"{ano_um} -> {melhor_pesos}")
        print("+-="*30)
        if 'model' in locals():
            del model
            print('DELETADO DPS DO WHILE')
        else:
            print("Nao consta model")
            pass
    else:
        print("PROblema no ANO: ", ano)
        print(res.solver.termination_condition)
        break



=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-
Anos totais:  ['2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025', '2026']
=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-
COMEÇANDO ANO NOVO:  2016
Nao consta model
Retornos atualizados
Sigmas Atualizados
-----
# ------ CRIAÇÃO DO MODELO
## Carteira criada para o ano:  2016
## UTILIZANDO SCORE DO ANO DE:  2015
## UTILIZANDO DADOS DE RETORNO DE:  2016
## UTILIZANDO SIGMAS DO ANO DE:  2016
Ano 2016 | escala var: 0.01065 | escala score: 0.5000
2016 -> {'ABEV3': 0.0, 'ALOS3': 0.0, 'ALPA4': 0.0, 'ANIM3': 0.0, 'AXIA3': 0.0, 'AZZA3': 0.0, 'B3SA3': 0.0, 'BEEF3': 0.0, 'CMIG4': 0.0, 'COGN3': 0.07572927032708582, 'CPFE3': 0.0, 'CPLE3': 0.0, 'CSAN3': 0.012441520930348657, 'CSMG3': 0.0, 'CSNA3': 0.0, 'CYRE3': 0.0, 'DIRR3': 0.0, 'ECOR3': 0.0, 'EGIE3': 0.0, 'EMBJ3': 0.0, 'ENEV3': 0.0, 'EQTL3': 0.0, 'EZTC3': 0.0, 'FLRY3': 0.15000

In [22]:
linhas = []
for an in anos:
    an = int(an.split('-')[0])
    print(an)
    for k, v in carteiras_anuais[an]['pesos'].items():
        if round(v, 4)>0.000 and v <= vb_peso_maximo*1.10:
            linhas.append({'ano': an, 'ativo': k, 'peso': v})

df_portfolios = pd.DataFrame(linhas)

# # visões instantâneas:
# df_portfolios[df_portfolios['ano'] == 2016].sort_values('peso', ascending=False)  # uma carteira
# df_portfolios.pivot(index='ativo', columns='ano', values='peso')                  # matriz ativo × ano
# df_portfolios.groupby('ativo')['ano'].count().sort_values(ascending=False)  


2016
2017
2018
2019
2020
2021
2022
2023
2024
2025
2026


In [23]:
df_portfolios[df_portfolios['ano'] == 2020].sort_values('peso', ascending=False)

,ano,ativo,peso
37,2020,CSMG3,0.15
39,2020,MOTV3,0.15
43,2020,WEGE3,0.15
38,2020,CSNA3,0.15
41,2020,SBSP3,0.15
42,2020,TIMS3,0.15
40,2020,SAPR11,0.10


In [25]:
df_portfolios.to_csv('../carteiras/min_var_piotroski/carteiras_minvar_piotroski.csv')
